In [ ]:

# Smart Study Planner & Task Manager
# DSA Mini Project - B.Tech CSE AI-ML

print("=" * 45)
print("       SMART STUDY PLANNER")
print("       DSA MINI PROJECT")
print("=" * 45)

print("Notebook setup successful!")


       SMART STUDY PLANNER
       DSA MINI PROJECT
Notebook setup successful!


In [ ]:

# Smart Study Planner: Dashboard Foundation

total_tasks = 10
completed_tasks = 4
pending_tasks = total_tasks - completed_tasks

progress = (completed_tasks / total_tasks) * 100

print("SMART STUDY PLANNER")
print("-" * 30)
print("Total Tasks:", total_tasks)
print("Completed Tasks:", completed_tasks)
print("Pending Tasks:", pending_tasks)
print("Study Progress:", progress, "%")


SMART STUDY PLANNER
------------------------------
Total Tasks: 10
Completed Tasks: 4
Pending Tasks: 6
Study Progress: 40.0 %


In [ ]:

# One study task stored in a dictionary

task1 = {
    "title": "Complete DSA Assignment",
    "subject": "DSA",
    "deadline": "2026-10-25",
    "priority": "High",
    "status": "Pending"
}

# Store tasks in a list
tasks = [task1]

# Display all tasks
for task in tasks:
    print("Task:", task["title"])
    print("Subject:", task["subject"])
    print("Deadline:", task["deadline"])
    print("Priority:", task["priority"])
    print("Status:", task["status"])
    print("-" * 30)


Task: Complete DSA Assignment
Subject: DSA
Deadline: 2026-10-25
Priority: High
Status: Pending
------------------------------


In [ ]:

# Smart Study Planner - Multiple Tasks

tasks = [
    {
        "title": "Complete DSA Assignment",
        "subject": "DSA",
        "priority": "High",
        "status": "Completed"
    },
    {
        "title": "Prepare DBMS Notes",
        "subject": "DBMS",
        "priority": "Medium",
        "status": "Completed"
    },
    {
        "title": "Practice Python Programs",
        "subject": "Python",
        "priority": "High",
        "status": "Pending"
    },
    {
        "title": "Revise Computer Networks",
        "subject": "CN",
        "priority": "Low",
        "status": "Completed"
    }
]

# Calculate dashboard statistics
total_tasks = len(tasks)
completed_tasks = 0

for task in tasks:
    if task["status"] == "Completed":
        completed_tasks += 1

pending_tasks = total_tasks - completed_tasks

if total_tasks > 0:
    progress = (completed_tasks / total_tasks) * 100
else:
    progress = 0

print("===== SMART STUDY PLANNER =====")
print("Total Tasks:", total_tasks)
print("Completed Tasks:", completed_tasks)
print("Pending Tasks:", pending_tasks)
print("Study Progress:", progress, "%")


===== SMART STUDY PLANNER =====
Total Tasks: 4
Completed Tasks: 3
Pending Tasks: 1
Study Progress: 75.0 %


In [ ]:

!pip install -q streamlit


In [ ]:

!streamlit run app.py &>/content/streamlit.log &


In [ ]:

!cat /content/streamlit.log




2026-10-27 13:22:22.621 Uvicorn server started on :::8501


In [ ]:

!pip install -q pyngrok


In [ ]:

from pyngrok import ngrok

ngrok.set_auth_token("3Kv72ZMdgRSvbXQqxmfIng3QA1k_e8PQo1VD4B2orE16gwLv")

public_url = ngrok.connect(8501)
print("Your Smart Study Planner:", public_url)


Your Smart Study Planner: NgrokTunnel: "https://domelike-freefall-exclusive.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:

!pkill -f "streamlit run app.py" || true


^C


In [ ]:

!streamlit run app.py --server.port 8501 &>/content/streamlit.log &


In [ ]:

%%writefile app.py
import heapq
from datetime import date
import streamlit as st
import json
from pathlib import Path
# File used to store tasks
TASKS_FILE = Path("tasks.json")
import uuid


def load_tasks():
    """Load tasks from the JSON file."""
    if TASKS_FILE.exists():
        try:
            with open(TASKS_FILE, "r", encoding="utf-8") as file:
                return json.load(file)
        except (json.JSONDecodeError, OSError):
            return []

    return []


def save_tasks(tasks):
    """Save tasks to the JSON file."""
    with open(TASKS_FILE, "w", encoding="utf-8") as file:
        json.dump(tasks, file, indent=4)
def recommend_tasks(tasks):
    """Recommend pending tasks using a priority queue."""

    priority_rank = {
        "High": 1,
        "Medium": 2,
        "Low": 3
    }

    task_heap = []

    for task in tasks:
        if task["completed"]:
            continue

        rank = priority_rank.get(task["priority"], 4)
        deadline = task["deadline"]

        # Smaller values are removed from the heap first.
        heapq.heappush(
            task_heap,
            (rank, deadline, task["id"], task)
        )

    recommendations = []

    while task_heap:
        _, _, _, task = heapq.heappop(task_heap)
        recommendations.append(task)

    return recommendations
# ==========================================
# 1. PAGE CONFIGURATION
# ==========================================
st.set_page_config(
    page_title="Smart Study Planner",
    page_icon="📚",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ==========================================
# 2. CUSTOM DARK THEME
# ==========================================
st.markdown("""
<style>

/* Main application background */
.stApp {
    background: linear-gradient(135deg, #0B1020 0%, #111827 100%);
    color: #F8FAFC;
}

/* Main content spacing */
.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
    max-width: 1200px;
}

/* Sidebar */
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #111827 0%, #1E1B4B 100%);
    border-right: 1px solid #312E81;
}

/* Sidebar text */
[data-testid="stSidebar"] p,
[data-testid="stSidebar"] label {
    color: #E2E8F0 !important;
}

/* Main heading */
.main-heading {
    font-size: 42px;
    font-weight: 800;
    background: linear-gradient(90deg, #A78BFA, #22D3EE);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 5px;
}

/* Subtitle */
.subtitle {
    color: #CBD5E1;
    font-size: 17px;
    margin-bottom: 25px;
}

/* Dashboard welcome card */
.welcome-card {
    background: linear-gradient(120deg, #1E1B4B, #172554);
    border: 1px solid #4F46E5;
    border-radius: 18px;
    padding: 28px;
    margin: 20px 0;
    box-shadow: 0 8px 25px rgba(0, 0, 0, 0.25);
}

.welcome-card h2 {
    color: #FFFFFF;
    margin-top: 0;
}

.welcome-card p {
    color: #CBD5E1;
    font-size: 16px;
}

/* Feature cards */
.feature-card {
    background: #161F32;
    border: 1px solid #293548;
    border-radius: 15px;
    padding: 22px;
    min-height: 165px;
    margin-bottom: 12px;
    transition: 0.3s ease;
}

.feature-card:hover {
    border-color: #8B5CF6;
    transform: translateY(-3px);
}

.feature-card h3 {
    color: #C4B5FD;
    font-size: 19px;
}

.feature-card p {
    color: #CBD5E1;
    font-size: 14px;
    line-height: 1.6;
}

/* Buttons */
.stButton > button {
    background: linear-gradient(90deg, #7C3AED, #6366F1);
    color: white;
    border: none;
    border-radius: 10px;
    padding: 10px 22px;
    font-weight: 600;
    transition: 0.2s ease;
}

.stButton > button:hover {
    background: linear-gradient(90deg, #6D28D9, #4F46E5);
    border: 1px solid #A78BFA;
    color: white;
}

/* Success messages */
[data-testid="stAlert"] {
    border-radius: 12px;
}

/* Divider */
hr {
    border-color: #293548;
}

/* General text */
.stApp p {
    color: #E2E8F0;
}

/* Footer */
.footer {
    text-align: center;
    color: #94A3B8;
    font-size: 13px;
    padding-top: 35px;
}

</style>
""", unsafe_allow_html=True)

# ==========================================
# 3. SIDEBAR NAVIGATION
# ==========================================
st.sidebar.markdown("## 📚 Study Planner")
st.sidebar.markdown("---")

page = st.sidebar.radio(
    "NAVIGATION",
    ["🏠 Dashboard", "📝 My Tasks", "📊 Progress"]
)

st.sidebar.markdown("---")
st.sidebar.caption("🎯 Plan smart. Study better.")
st.sidebar.caption("Your journey to academic success.")

# ==========================================
# 4. DASHBOARD PAGE
# ==========================================
if page == "🏠 Dashboard":

    st.markdown(
        '<div class="main-heading">📚 Smart Study Planner</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">'
        'Your personal academic productivity dashboard'
        '</div>',
        unsafe_allow_html=True
    )

    st.markdown("""
    <div class="welcome-card">
        <h2>✨ Welcome to your study space!</h2>
        <p>
        Organize your assignments, manage deadlines, and track
        your academic progress — all in one place.
        </p>
    </div>
    """, unsafe_allow_html=True)

    # Dashboard feature cards
    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("""
        <div class="feature-card">
            <h3>📝 Manage Tasks</h3>
            <p>
            Keep your assignments and study activities organized
            so you always know what to do next.
            </p>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        st.markdown("""
        <div class="feature-card">
            <h3>⏰ Track Deadlines</h3>
            <p>
            Stay aware of upcoming deadlines and make time
            for the subjects that need your attention.
            </p>
        </div>
        """, unsafe_allow_html=True)

    with col3:
        st.markdown("""
        <div class="feature-card">
            <h3>📊 Monitor Progress</h3>
            <p>
            Build consistent study habits and keep track
            of your academic productivity.
            </p>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")

    st.subheader("🚀 Ready to get started?")

    st.write(
        "Head over to **My Tasks** from the sidebar to begin "
        "organizing your study schedule."
    )

    if st.button("✨ Let's Get Started"):
        st.info("Choose **📝 My Tasks** from the sidebar to continue!")

    st.markdown("""
    <div class="footer">
        Built with Python and Streamlit | Smart Study Planner 📚
    </div>
    """, unsafe_allow_html=True)

# ==========================================
# 5. MY TASKS PAGE — PLACEHOLDER
# ==========================================
elif page == "📝 My Tasks":
    st.title("📝 My Tasks")
    st.write("Add assignments, organize deadlines, and manage your workload.")

    # Initialize task storage
    if "tasks" not in st.session_state:
      st.session_state.tasks = load_tasks()

    st.markdown("### ➕ Add a New Task")

    with st.form("task_form"):
        task_title = st.text_input("Task Name", placeholder="e.g., Complete Python assignment")

        subject = st.selectbox(
            "Subject",
            [
                "Python",
                "Data Structures & Algorithms",
                "DBMS",
                "Computer Networks",
                "Mathematics",
                "Other"
            ]
        )

        deadline = st.date_input("Deadline")

        priority = st.selectbox(
            "Priority",
            ["High", "Medium", "Low"]
        )

        submitted = st.form_submit_button("➕ Add Task")

        if submitted:
            if task_title.strip():
                task = {
                    "id": str(uuid.uuid4()),
                    "title": task_title.strip(),
                    "subject": subject,
                    "deadline": str(deadline),
                    "priority": priority,
                    "completed": False
                }

                st.session_state.tasks.append(task)
                save_tasks(st.session_state.tasks)
                st.success("Task added successfully!")
            else:
                st.warning("Please enter a task name.")

    st.divider()
    st.markdown("### 📋 Your Task List")
        # Search tasks
    search_query = st.text_input(
        "🔍 Search Tasks",
        placeholder="Enter a task name..."
    )

    # Filter options
    filter_col1, filter_col2, filter_col3 = st.columns(3)

    with filter_col1:
        subject_filter = st.selectbox(
            "Filter by Subject",
            ["All"] + sorted(set(task["subject"] for task in st.session_state.tasks)),
            key="subject_filter"
        )

    with filter_col2:
        priority_filter = st.selectbox(
            "Filter by Priority",
            ["All", "High", "Medium", "Low"],
            key="priority_filter"
        )

    with filter_col3:
        status_filter = st.selectbox(
            "Filter by Status",
            ["All", "Pending", "Completed"],
            key="status_filter"
        )

    # Apply filters
    filtered_tasks = st.session_state.tasks.copy()

    if search_query.strip():
        filtered_tasks = [
            task for task in filtered_tasks
            if search_query.casefold() in task["title"].casefold()
        ]

    if subject_filter != "All":
        filtered_tasks = [
            task for task in filtered_tasks
            if task["subject"] == subject_filter
        ]

    if priority_filter != "All":
        filtered_tasks = [
            task for task in filtered_tasks
            if task["priority"] == priority_filter
        ]

    if status_filter != "All":
        is_completed = status_filter == "Completed"
        filtered_tasks = [
            task for task in filtered_tasks
            if task["completed"] == is_completed
        ]

    st.caption(f"Showing {len(filtered_tasks)} of {len(st.session_state.tasks)} tasks.")
    sort_option = st.selectbox(
        "↕️ Sort Tasks By",
        [
            "Default Order",
            "Nearest Deadline",
            "Highest Priority",
            "Task Name (A–Z)"
        ],
        key="task_sort"
    )

    priority_order = {"High": 1, "Medium": 2, "Low": 3}

    if sort_option == "Nearest Deadline":
        filtered_tasks.sort(key=lambda task: task["deadline"])

    elif sort_option == "Highest Priority":
        filtered_tasks.sort(
            key=lambda task: (
                priority_order.get(task["priority"], 4),
                task["deadline"]
            )
        )

    elif sort_option == "Task Name (A–Z)":
        filtered_tasks.sort(key=lambda task: task["title"].casefold())
    st.divider()
    st.markdown("### 🧠 Smart Task Recommendations")
    st.write("Your suggested order of work, based on priority and deadline.")

    recommended = recommend_tasks(st.session_state.tasks)

    if recommended:
        for index, task in enumerate(recommended, start=1):
            st.markdown(
                f"**{index}. {task['title']}** — "
                f"{task['priority']} Priority | "
                f"Deadline: {task['deadline']}"
            )
    else:
        st.success("You're all caught up! No pending tasks.")
    if not st.session_state.tasks:
        st.info("No tasks yet. Add your first task above!")
    else:
        for task in filtered_tasks:
            with st.container(border=True):
                col1, col2 = st.columns([4, 1])

                with col1:
                    st.markdown(f"**{task['title']}**")
                    st.write(f"📚 Subject: {task['subject']}")
                    st.write(f"📅 Deadline: {task['deadline']}")
                    st.write(f"⚡ Priority: {task['priority']}")

                    if task["completed"]:
                        st.success("Completed")
                    else:
                        st.warning("Pending")

                with col2:
                    if not task["completed"]:
                        if st.button("✅ Complete", key=f"complete_{task['id']}"):
                            task["completed"] = True
                            save_tasks(st.session_state.tasks)
                            st.rerun()

                    if st.button("🗑️ Delete", key=f"delete_{task['id']}"):
                        st.session_state.tasks.remove(task)
                        save_tasks(st.session_state.tasks)
                        st.rerun()

# ==========================================
elif page == "📊 Progress":
    import pandas as pd
    from datetime import date

    st.title("📊 Progress Dashboard")
    st.write("Track your academic productivity and deadlines.")

    # Load tasks if they have not been initialized
    if "tasks" not in st.session_state:
        st.session_state.tasks = load_tasks()

    tasks = st.session_state.tasks

    # Calculate statistics
    total_tasks = len(tasks)
    completed_tasks = sum(1 for task in tasks if task["completed"])
    pending_tasks = total_tasks - completed_tasks

    today = date.today()
    overdue_tasks = sum(
        1 for task in tasks
        if not task["completed"]
        and date.fromisoformat(task["deadline"]) < today
    )

    completion_percentage = (
        (completed_tasks / total_tasks) * 100
        if total_tasks > 0 else 0
    )

    # Display metrics
    col1, col2, col3, col4 = st.columns(4)

    col1.metric("📚 Total Tasks", total_tasks)
    col2.metric("✅ Completed", completed_tasks)
    col3.metric("⏳ Pending", pending_tasks)
    col4.metric("🚨 Overdue", overdue_tasks)

    st.divider()

    # Overall progress
    st.subheader("🎯 Overall Completion")
    st.progress(completion_percentage / 100)
    st.write(f"**{completion_percentage:.1f}%** of your tasks completed.")

    st.divider()

    # Task completion chart
    st.subheader("📈 Task Status Distribution")

    if total_tasks > 0:
        chart_data = pd.DataFrame({
            "Status": ["Completed", "Pending"],
            "Tasks": [completed_tasks, pending_tasks]
        })

        st.bar_chart(chart_data, x="Status", y="Tasks")

    else:
        st.info("No tasks available yet. Add tasks to see your progress!")

    st.divider()

    # Overdue task details
    st.subheader("🚨 Overdue Tasks")

    overdue_list = [
        task for task in tasks
        if not task["completed"]
        and date.fromisoformat(task["deadline"]) < today
    ]

    if overdue_list:
        for task in overdue_list:
            st.error(
                f"**{task['title']}** — "
                f"Deadline: {task['deadline']} | "
                f"Priority: {task['priority']}"
            )



Overwriting app.py


In [ ]:

!streamlit run app.py --server.port 8501 &>/content/streamlit.log &


In [ ]:
!cat tasks.json

[
    {
        "id": 1,
        "title": "Make DSA assignment",
        "subject": "Python",
        "deadline": "2026-10-27",
        "priority": "High",
        "completed": false
    },
    {
        "id": 2,
        "title": "Make DSA assignment",
        "subject": "Data Structures & Algorithms",
        "deadline": "2026-10-27",
        "priority": "High",
        "completed": true
    },
    {
        "id": 3,
        "title": "Prepare for Maths PPt",
        "subject": "Mathematics",
        "deadline": "2026-10-28",
        "priority": "High",
        "completed": false
    },
    {
        "id": 4,
        "title": "CN file",
        "subject": "Other",
        "deadline": "2026-10-28",
        "priority": "Medium",
        "completed": false
    }
]